# Paso 1/2 extendido - KPCL0035 (agua), escrutinio period-over-period

KPCL0034 tuvo un escrutinio completo abril-vs-mayo+ porque tiene 2 UUID que
delimitan el corte. **KPCL0035 tiene un solo UUID** desde que se creo
(2026-05-25) asi que nunca recibio la misma pregunta: ¿es su fondo estable en
el tiempo, o hay un antes/despues real escondido en un solo UUID continuo?

## Corte elegido: el apagon real de KPCL0035, verificado en la data (no en el doc)

`Knowledge/09_Sensores/README_Sensores.md` documenta un solo apagon
("se apago junto con KPCL0034 el 23-jul-2026 pero se reconecto solo el
10-ago-2026"). Verificando directo contra `readings`/`readings_rows` (no
confiando en el doc): el gap real es mas largo y tiene una forma distinta -

```
2026-05-25 -------- 2026-06-27 04:50   [~26 dias sin datos]   2026-07-23 01:07-05:37
   Periodo A                                                    (blip de ~4.5h, se ignora)
   (pre-apagon)                                                          |
                                                          [~18 dias sin datos]
                                                                          |
                                                                 2026-08-10 12:33 -------- 2026-08-28
                                                                          Periodo B (post-reconexion)
```

Cortes usados: `Periodo A` = todo antes de 2026-06-27 12:00 UTC, `Periodo B` =
todo desde 2026-08-10 00:00 UTC. El blip de ~4.5h del 23-jul queda excluido de
ambos periodos (muy chico para aportar, y esta justo en la frontera del gap).

Carga desde `data/lecturas_limpias.csv` (cache de Paso 1), filtrado a
`device_code == "KPCL0035"` -- **correr `01_caracterizacion_fondo.ipynb`
primero** si el cache no existe todavia.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CACHE_CSV = NOTEBOOK_DIR / "data" / "lecturas_limpias.csv"
GAP_CUTOFF_S = 300

# --- Carga desde cache (post-dedup, generado por 01_caracterizacion_fondo.ipynb) ---
# Si no existe, correr ese notebook primero -- el dedup de abril vive ahi (no le
# aplica a KPCL0035, que nunca tuvo ese patron -- Paso 1 bloque 1).
df = pd.read_csv(CACHE_CSV)
df = df[df["device_code"] == "KPCL0035"].drop(columns=["device_id", "device_code"])
df["ts"] = pd.to_datetime(df["ts"], format="ISO8601", utc=True)
df = df.sort_values("ts").reset_index(drop=True)

# --- Cortes verificados contra la data real (ver celda anterior) ------------
CORTE_APAGON = pd.Timestamp("2026-06-27 12:00:00", tz="UTC")
CORTE_RECONEXION = pd.Timestamp("2026-08-10 00:00:00", tz="UTC")
df["periodo"] = np.select(
    [df["ts"] < CORTE_APAGON, df["ts"] >= CORTE_RECONEXION],
    ["A_pre_apagon", "B_post_reconexion"],
    default="blip_23jul_excluido",
)

print(df["periodo"].value_counts())
for _p in ("A_pre_apagon", "B_post_reconexion"):
    _sub = df[df["periodo"] == _p]
    print(f"{_p}: {_sub['ts'].min()} -> {_sub['ts'].max()}")

# delta_t/delta_peso calculados DENTRO de cada periodo -- nunca cruzando el gap
df["delta_peso"] = df.groupby("periodo", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("periodo", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
df["velocidad_peso"] = df["delta_peso"].where(df["delta_t"] <= GAP_CUTOFF_S) / df["delta_t"]

# chequeo de dedup (mismo criterio de Paso 1) -- ya se sabia que KPCL0035 no
# lo necesitaba (fraccion <1s ~0.006% en Paso 1, bloque 1); se re-verifica aca
# por periodo para no asumirlo sin mirar
for _p in ("A_pre_apagon", "B_post_reconexion"):
    _sub = df[df["periodo"] == _p]
    print(f"{_p}: cadencia mediana {_sub['delta_t'].median():.2f}s, "
          f"fraccion delta_t<1s: {(_sub['delta_t'] < 1).mean()*100:.4f}%")


## Nivel: valle (P05 semanal) vs. pico (P95 semanal) por periodo

Misma logica que Paso 1 Hallazgo 2: el valle es proxy de tara/hardware
(deberia ser constante si nada cambio fisicamente), el pico es proxy de
cuanta agua hay servida (varia con habito de relleno).


In [ ]:
df_idx = df.set_index("ts")
filas_nivel = []
for _p in ("A_pre_apagon", "B_post_reconexion"):
    _sub = df_idx[df_idx["periodo"] == _p]["peso"]
    _semanal = _sub.resample("W").agg(["count", lambda s: s.quantile(.05), lambda s: s.quantile(.95)])
    _semanal.columns = ["n", "p05", "p95"]
    _semanal = _semanal[_semanal["n"] > 500]  # semanas parciales de borde, poco representativas
    filas_nivel.append({
        "periodo": _p,
        "valle_p05_mediana_semanal_g": round(_semanal["p05"].median(), 1),
        "pico_p95_mediana_semanal_g": round(_semanal["p95"].median(), 1),
        "n_semanas": len(_semanal),
    })

nivel_df = pd.DataFrame(filas_nivel).set_index("periodo")
nivel_df["diferencia_vs_A_valle_g"] = nivel_df["valle_p05_mediana_semanal_g"] - nivel_df["valle_p05_mediana_semanal_g"].iloc[0]
nivel_df["diferencia_vs_A_pico_g"] = nivel_df["pico_p95_mediana_semanal_g"] - nivel_df["pico_p95_mediana_semanal_g"].iloc[0]
nivel_df


## Hallazgo 5 - a diferencia de KPCL0034, el valle SI se movio

**Resultado real (2026-08-29):**

| | Valle (P05 semanal) | Pico (P95 semanal) |
|---|---|---|
| Periodo A - pre-apagon | 319.0g | 542.0g |
| Periodo B - post-reconexion | 411.0g | 570.0g |
| Diferencia | **+92g** | +28g |

En KPCL0034 el valle casi no se movia (~8g) y el pico se movia 3x mas - la
conclusion era "apetito/servido, no tara". **Aca es al reves**: el valle se
mueve mas que el pico (+92g vs +28g). Aplicando la misma logica de
diagnostico (valle = proxy de tara/hardware, pico = proxy de cuanto se sirvio),
esto **si apunta a un cambio de tara/hardware o del recipiente fisico** entre
los dos periodos, no a un cambio de consumo de agua.

Candidatos sin confirmar (fuera de alcance verificarlos aca, requieren info
operacional que no esta en `readings`): el dispositivo estuvo apagado/
desconectado 18 dias completos antes de reconectarse el 10-ago - suficiente
tiempo para que alguien haya cambiado el recipiente de agua, re-tarado el
sensor manualmente, o modificado cuanto liquido se deja de base. **No se
puede asumir 'mismo fondo' para KPCL0035 entre estos dos periodos** sin mas
investigacion - a diferencia de KPCL0034, donde si se pudo descartar tara.


## Dinamica y duracion de estabilidad por periodo


In [ ]:
filas_dinamica = []
for _p in ("A_pre_apagon", "B_post_reconexion"):
    _sub = df[df["periodo"] == _p]
    _abs_delta = _sub["abs_delta_peso"].dropna()
    _vel = _sub["velocidad_peso"].abs().dropna()

    _is_gap = _sub["delta_t"] > GAP_CUTOFF_S
    _paso_estable = (_sub["delta_peso"] == 0) & (~_is_gap.fillna(False))
    _corrida_id = (~_paso_estable).cumsum()
    _duraciones = _sub["delta_t"][_paso_estable].groupby(_corrida_id[_paso_estable]).sum()

    filas_dinamica.append({
        "periodo": _p,
        "abs_delta_p50_g": round(_abs_delta.quantile(.50), 3),
        "abs_delta_p90_g": round(_abs_delta.quantile(.90), 3),
        "abs_delta_p95_g": round(_abs_delta.quantile(.95), 3),
        "abs_delta_p99_g": round(_abs_delta.quantile(.99), 3),
        "velocidad_p95_g_s": round(_vel.quantile(.95), 4),
        "velocidad_p99_g_s": round(_vel.quantile(.99), 4),
        "duracion_p50_s": round(_duraciones.quantile(.50), 1),
        "duracion_p90_s": round(_duraciones.quantile(.90), 1),
        "duracion_p95_s": round(_duraciones.quantile(.95), 1),
    })

dinamica_df = pd.DataFrame(filas_dinamica).set_index("periodo")
dinamica_df


## Cierre - KPCL0035 tambien queda con baseline separado por periodo

| Pregunta | Resultado |
|---|---|
| ¿Duplicado de escritura tipo abril-KPCL0034? | No en ninguno de los dos periodos - cadencia mediana ~30s, fraccion `Δt<1s` despreciable en ambos |
| ¿Tara/hardware distinto entre periodos? | **Posible que si** - el valle (P05 semanal) se movio +92g, mas que el pico (+28g). Sin confirmar la causa fisica |
| ¿Cuerpo de la dinamica (`\|Δpeso\|`/velocidad) comparable? | A confirmar con los numeros de la celda anterior - no asumir que si sin mirarlos |
| ¿Duracion de estabilidad comparable? | A confirmar con los numeros de la celda anterior |

**Diferencia clave con KPCL0034:** ahi se pudo descartar tara/hardware con
evidencia (valle estable). Aca **no se puede descartar** con la misma
confianza - el valle es la variable que mas se movio. El baseline de KPCL0035
queda separado por periodo (A: pre-apagon, B: post-reconexion) hasta que se
confirme o descarte un cambio fisico real en el recipiente/tara durante los 18
dias que estuvo desconectado.


## Resumen final (tabla y gráfico)

Une `nivel_df` y `dinamica_df` (ya calculados arriba) en una sola tabla, y
grafica valle/pico y duración de estabilidad lado a lado por período.


In [ ]:
import matplotlib.pyplot as plt

resumen_final_df = nivel_df.join(dinamica_df)
print("--- Resumen final KPCL0035 (period-over-period) ---")
print(resumen_final_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
_x = np.arange(len(resumen_final_df))

# Panel 1: valle vs pico por periodo
axes[0].bar(_x - 0.2, resumen_final_df["valle_p05_mediana_semanal_g"], width=0.4, label="Valle (P05)", color="#2980b9")
axes[0].bar(_x + 0.2, resumen_final_df["pico_p95_mediana_semanal_g"], width=0.4, label="Pico (P95)", color="#27ae60")
axes[0].set_xticks(_x)
axes[0].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[0].set_ylabel("Peso semanal (g)")
axes[0].set_title("Valle vs pico por periodo")
axes[0].legend()

# Panel 2: dinamica -- abs_delta_p99
axes[1].bar(resumen_final_df.index, resumen_final_df["abs_delta_p99_g"], color="#c0392b")
axes[1].set_xticks(np.arange(len(resumen_final_df)))
axes[1].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[1].set_ylabel("|delta_peso| P99 (g)")
axes[1].set_title("Dinamica por periodo")

# Panel 3: duracion de estabilidad
axes[2].bar(_x - 0.2, resumen_final_df["duracion_p90_s"], width=0.4, label="P90", color="#8e44ad")
axes[2].bar(_x + 0.2, resumen_final_df["duracion_p95_s"], width=0.4, label="P95", color="#e67e22")
axes[2].set_xticks(_x)
axes[2].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[2].set_ylabel("Duracion estable (s)")
axes[2].set_title("Duracion de estabilidad por periodo")
axes[2].legend()

fig.suptitle("KPCL0035 - Resumen: nivel y dinamica pre/post apagon")
fig.tight_layout()
plt.show()
